# Handling Missing Values

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('housing_data_after_text_feature_eng.csv').drop(columns={'Unnamed: 0'})

In [ ]:
df.head()

,url,image_url,beds,baths,sqft,address,property_type,region,parking_total_spaces,walk_score,...,property_age,mobility_score,avg_school_distance_miles,parking_quality_score,has_garage,county,zip_price_tier,price_volatility,price_reduction_total,price
0,https://www.zillow.com/homedetails/197-Lowell-...,https://photos.zillowstatic.com/fp/90454dcdc33...,3.0,2.0,1415.0,"197 Lowell St, Arlington, MA 02474",single family,Arlington,4.0,81.0,...,145.0,73.0,0.900,0.0,0,Middlesex,Mid-tier,190775.778337,481500.0,999000
1,https://www.zillow.com/homedetails/225-Mystic-...,https://photos.zillowstatic.com/fp/e92a0b0b33e...,2.0,1.0,1064.0,"225 Mystic St, Arlington, MA 02474",single family,Arlington,3.0,69.0,...,145.0,65.0,0.300,1.0,0,Middlesex,Mid-tier,75258.487893,189000.0,799000
2,https://www.zillow.com/homedetails/43-Longmead...,https://photos.zillowstatic.com/fp/740e93bfcff...,2.0,1.0,1628.0,"43 Longmeadow Rd, Arlington, MA 02474",single family,Arlington,5.0,32.0,...,73.0,33.2,3.400,0.0,0,Middlesex,Mid-tier,379500.000000,759000.0,899000
3,https://www.zillow.com/homedetails/11-Pine-Ct-...,https://photos.zillowstatic.com/fp/bb84e595f96...,3.0,2.0,1824.0,"11 Pine Ct, Arlington, MA 02476",single family,Arlington,2.0,84.0,...,99.0,72.0,0.700,0.0,0,Middlesex,Mid-tier,358034.448994,815000.0,998000
4,https://www.zillow.com/homedetails/17-Norcross...,https://photos.zillowstatic.com/fp/ac474eb496d...,2.0,2.0,1221.0,"17 Norcross St FLOOR 3, Arlington, MA 02474",condo,Arlington,1.0,77.0,...,117.0,84.6,1.733,1.0,0,Middlesex,Mid-tier,16004.999219,-47000.0,668000


In [ ]:
for i in df.columns:
  if (df[i].isnull().sum()) != 0:
    print( f"{i} -> {df[i].isnull().sum()}")

beds -> 96
baths -> 87
sqft -> 103
address -> 33
parking_total_spaces -> 1026
walk_score -> 333
bike_score -> 534
elementary_school_distance -> 2460
middle_school_distance -> 22
wind_risk -> 772
sqft_lot_cleaned -> 1967
zip_code -> 149
property_age -> 30
mobility_score -> 535


In [ ]:
# Drop features with excessive missing that are hard to impute meaningfully
df.drop(['elementary_school_distance'], axis=1, inplace=True)  # 2460 missing, have avg_school_distance

# Strategic imputation:
# Parking - use property context
df['parking_total_spaces'] = df['parking_total_spaces'].fillna(
    df.groupby(['has_garage', 'property_type'])['parking_total_spaces'].transform('median')
)

# Geographic features - use location-based imputation
df['wind_risk'] = df['wind_risk'].fillna(df.groupby('county')['wind_risk'].transform('median'))

# Simple mean/median only for truly random missing:
df['beds'] = df['beds'].fillna(df['beds'].median())
df['baths'] = df['baths'].fillna(df['baths'].median())
df['sqft'] = df['sqft'].fillna(df['sqft'].median())
df['property_age'] = df['property_age'].fillna(df['property_age'].median())

# Consider dropping sqft_lot_cleaned (1967 missing = 25%) unless lot size is crucial

In [ ]:
for i in df.columns:
  if (df[i].isnull().sum()) != 0:
    print( f"{i} -> {df[i].isnull().sum()}")

address -> 33
walk_score -> 333
bike_score -> 534
middle_school_distance -> 22
sqft_lot_cleaned -> 1967
zip_code -> 149
mobility_score -> 535


In [ ]:
# Drop non-predictive and redundant features
df.drop(['bike_score', 'sqft_lot_cleaned'], axis=1, inplace=True)

# Location-based imputation
df['walk_score'] = df['walk_score'].fillna(df.groupby('county')['walk_score'].transform('median'))
df['mobility_score'] = df['mobility_score'].fillna(df.groupby('county')['mobility_score'].transform('median'))

# Simple median for school distance (only 22 missing)
df['middle_school_distance'] = df['middle_school_distance'].fillna(df['middle_school_distance'].median())

# Handle zip_code missing - this is categorical
# Either drop these rows (149 out of 7941 = 1.9%) or use county as proxy
df = df.dropna(subset=['zip_code'])  # Clean approach since it's small percentage

In [ ]:
for i in df.columns:
  if (df[i].isnull().sum()) != 0:
    print( f"{i} -> {df[i].isnull().sum()}")

In [ ]:
df.shape

(7792, 25)

In [ ]:
df.sample(10)

,url,image_url,beds,baths,sqft,address,property_type,region,parking_total_spaces,walk_score,...,property_age,mobility_score,avg_school_distance_miles,parking_quality_score,has_garage,county,zip_price_tier,price_volatility,price_reduction_total,price
943,https://www.zillow.com/homedetails/144-South-S...,https://photos.zillowstatic.com/fp/82513c5d22e...,3.0,1.0,1482.0,"144 South St, Tewksbury, MA 01876",single family,Tewksbury,4.0,28.0,...,95.0,32.0,2.133,1.0,0,Middlesex,Mid-tier,187950.000000,375900.0,399900
4821,https://www.zillow.com/homedetails/314-Grovela...,https://photos.zillowstatic.com/fp/31fb98a77ca...,4.0,3.0,2442.0,"314 Groveland St, Abington, MA 02351",single family,Abington,6.0,28.0,...,68.0,30.8,1.367,1.0,0,Plymouth,Mid-tier,181340.270210,350400.0,699900
5729,https://www.zillow.com/homedetails/3-Blanchard...,https://photos.zillowstatic.com/fp/028ceb1934b...,3.0,3.0,1330.0,"3 Blanchard St, Avon, MA 02322",single family,Avon,2.0,17.0,...,44.0,18.6,2.733,2.5,1,Norfolk,Mid-tier,127542.908858,310000.0,639900
7503,https://www.zillow.com/homedetails/30-Kathy-Tr...,https://photos.zillowstatic.com/fp/5ad18817b7d...,4.0,2.0,2014.0,"30 Kathy Trl, Feeding Hills, MA 01030",single family,Feeding Hills,6.0,3.0,...,50.0,12.2,3.500,1.0,0,Hampden,Budget,0.000000,0.0,299900
4922,https://www.zillow.com/homedetails/3030-Cranbe...,https://photos.zillowstatic.com/fp/d02f33d8d92...,1.0,1.0,600.0,"3030 Cranberry Hwy LOT 13, East Wareham, MA 02538",condo,East Wareham,2.0,53.0,...,47.0,47.0,5.600,1.0,0,Plymouth,Budget,33968.220442,81500.0,119000
759,https://www.zillow.com/homedetails/783-Cambrid...,https://photos.zillowstatic.com/fp/d100b871708...,1.0,1.0,473.0,"783 Cambridge St APT 2, Cambridge, MA 02141",condo,Cambridge,1.0,96.0,...,125.0,95.6,0.933,1.0,0,Middlesex,High-end,29500.000000,59000.0,479000
4348,https://www.zillow.com/homedetails/39-Bennets-...,https://photos.zillowstatic.com/fp/6f53bfdecb1...,3.0,2.0,1864.0,"39 Bennets Neck Dr, Bourne, MA 02532",single family,Bourne,3.0,9.0,...,49.0,11.4,2.167,1.5,1,Barnstable,Mid-tier,7500.000000,-15000.0,712000
6340,https://www.zillow.com/homedetails/10-Garrison...,https://photos.zillowstatic.com/fp/25c0bb589f1...,6.0,5.0,5267.0,"10 Garrison St, Groveland, MA 01834",single family,Groveland,30.0,32.0,...,322.0,23.6,1.400,8.5,1,Essex,Mid-tier,325908.985250,665000.0,1100000
3014,https://www.zillow.com/homedetails/49-Snow-Cre...,https://photos.zillowstatic.com/fp/eeed60f72d1...,3.0,2.0,2444.0,"49 Snow Creek Dr, Barnstable, MA 02630",single family,Barnstable,6.0,51.0,...,51.0,46.2,2.867,2.5,1,Barnstable,Mid-tier,2500.000000,-5000.0,794900
4483,https://www.zillow.com/homedetails/43-Salt-Mar...,https://photos.zillowstatic.com/fp/ceb1d1cadce...,4.0,3.0,2708.0,"43 Salt Marsh Ln, Plymouth, MA 02360",single family,Plymouth,8.0,8.0,...,90.0,9.2,4.000,1.0,0,Plymouth,Mid-tier,248190.963574,449100.0,1299000


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7792 entries, 0 to 7940
Data columns (total 25 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   url                        7792 non-null   object 
 1   image_url                  7792 non-null   object 
 2   beds                       7792 non-null   float64
 3   baths                      7792 non-null   float64
 4   sqft                       7792 non-null   float64
 5   address                    7792 non-null   object 
 6   property_type              7792 non-null   object 
 7   region                     7792 non-null   object 
 8   parking_total_spaces       7792 non-null   float64
 9   walk_score                 7792 non-null   float64
 10  middle_school_distance     7792 non-null   float64
 11  wind_risk                  7792 non-null   float64
 12  nearby_cities              7792 non-null   object 
 13  sqft_lot_category          7792 non-null   object 
 1

In [ ]:
df.to_csv('housing_data_after_handling_missing_value.csv', index=False)